---
title: Assignment-4
author: Siddharth Rajesh
date: 19-03-2026
---

### Importing libraries

In [1]:
import numpy as np
import pandas as pd

## Loading the counts matrix 

In [2]:
counts = pd.read_table('data/argR-counts-matrix.txt', header=None, index_col=0)     # Load the counts matrix as a dataframe with the first column set to be the index

counts_matrix = counts.iloc[:, 1:].to_numpy()       # Extracting the counts alone and converting it into a numpy ndarray

## Defining a function to generate the position-specific frequency matrix

In [3]:
def compute_frequency_matrix(counts_matrix):
    adjusted_counts_matrix = counts_matrix + 1      # Adjust the counts matrix by 1 to prevent inf value when performing log-transformation
    total_number_of_sequences = np.sum(adjusted_counts_matrix, axis = 0)    # Calculate the total number of sequences used to build the counts matrix
    frequency_matrix = adjusted_counts_matrix / total_number_of_sequences   # Divide each of the pseudocount by the total number of sequences to generate the frequency of each base at that specific position
    return frequency_matrix                     # Return the position-specific frequency matrix

## Function for computing the Position-specific scoring matrix

In [4]:
def compute_scoring_matrix(frequency_matrix):
    background_frequency = 0.25         # Background frequency is 0.25, since there is 1/4 probability for the position to be a specific nucleotide amongst the 4 nucleotides
    scoring_matrix = np.log2(frequency_matrix / background_frequency)       # Calculating the position-specific scoring matrix by dividing each frequency by the background frequency and then applying log2-transformation
    return scoring_matrix       # Returning the position-specific scoring matrix

## Computing the Position-specific Scoring Matrix

In [5]:
frequency_matrix = compute_frequency_matrix(counts_matrix)
scoring_matrix = compute_scoring_matrix(frequency_matrix)

score_df = pd.DataFrame(scoring_matrix, index=['A', 'C', 'G', 'T'])     # Converting the scoring matrix into a pandas dataframe
score_df.columns = [f'Pos {i+1}' for i in range(score_df.shape[1])]
score_df.to_csv('results/scoring_matrix.csv')

print('Scoring Matrix:')
score_df.style.format('{:.3f}').background_gradient(cmap='RdYlGn', axis=None)       # Printing the scoring matrix in the form of a heatmap

Scoring Matrix:


,Pos 1,Pos 2,Pos 3,Pos 4,Pos 5,Pos 6,Pos 7,Pos 8,Pos 9,Pos 10,Pos 11,Pos 12,Pos 13,Pos 14,Pos 15,Pos 16,Pos 17,Pos 18
A,0.216,0.746,1.505,0.368,-0.632,-1.369,1.505,1.505,-0.954,0.505,0.216,-0.369,0.046,1.746,-0.632,-1.369,-1.369,1.746
C,0.046,-0.632,-1.954,-0.147,-1.369,-0.954,-0.954,-0.954,-1.954,-2.954,-1.369,-2.954,0.046,-2.954,-0.954,-0.954,1.690,-2.954
G,-0.954,-1.369,-1.954,0.216,-1.369,1.505,-1.369,-1.369,-2.954,-1.954,-2.954,-1.954,-2.954,-1.954,-2.954,1.046,-2.954,-1.369
T,0.368,0.368,-0.632,-0.632,1.368,-1.954,-1.954,-1.954,1.631,1.133,1.216,1.505,0.853,-1.954,1.438,0.046,-1.954,-2.954


## Function for reading the sequences from the upstream regions file

In [6]:
def read_sequence(file_path):
    sequences = {}                  # Dictionary to store the GeneID: Sequence
    with open(file_path, 'r') as f:
        for line in f:                  # Iterating over each line in the file
            line = line.strip()         # Removing any trailing whitespaces
            if not line:
                continue
            parts = line.split(' \ ')
            sequences[parts[0]] = parts[1]     # First column is GeneID and third column contains the sequences
    return sequences                    # Returning the dictionary containing gene_id and sequences

## Function for scoring the motif

In [7]:
def score_motif(motif, scoring_matrix):
    base_to_row = {
        'A': 0,
        'C': 1,
        'G': 2,
        'T': 3
    }                   # Map the base to the index of the counts matrix
    
    score = 0
    for i, base in enumerate(motif.upper()):        # Iterate over every base
        if base in base_to_row.keys():              # If the base is in the mapping dict, we execute the conditional
            score += scoring_matrix[base_to_row[base], i]       # Add the score from the scoring matrix to the current score of the motif
    return score                                    # Return the scores

## Wrapper function for computing the top 30 highest-scoring motifs from the given sequences

In [8]:
def score_sequences(sequences, scoring_matrix, top_k = 30):
    top_scores = []                 # List to store the results
    for gene_id, sequence in sequences.items():     # Iterate over the sequences dict
        for i in range(len(sequence) - len(scoring_matrix[0]) + 1):         # Sliding window approach over each seqeuence in the dictionary
            motif = sequence[i : i+len(scoring_matrix[0])]                  # Extract the motif
            score = score_motif(motif, scoring_matrix)                      # Score the motiff using the computed scoring matrix
            top_scores.append((gene_id, motif, score))                      # Append the gene_id, motif, and the score for the motif
    
    top_scores.sort(key=lambda x: x[2], reverse=True)                       # Sort the motifs based on their scores in descending order
    return top_scores[:top_k]                                               # Return the top_k requested motifs and their scores

## Loading the sequences and calculating the top scoring motifs

In [9]:
sequences = read_sequence('data/E_coli_K12_MG1655.400_50')          # Call the function to get the sequences
top_scoring_motifs = score_sequences(sequences, scoring_matrix)     # Score the motifs in each sequence

## Printing the top 30 motifs with highest scores

In [10]:
print("Top 30 scoring motifs:")  # Print the top 30 scoring motifs
for gene_id, motif, score in top_scoring_motifs:  # Iterate over each top-scoring motifs.
    print(f"Gene ID: {gene_id}, Score: {score}")  # Print the gene ID, motif, and its score.

Top 30 scoring motifs:
Gene ID: b3171, Score: 20.981522027249504
Gene ID: 16128258, Score: 20.76324571438529
Gene ID: 16132076, Score: 19.915248807830338
Gene ID: 16131126, Score: 19.093247109808335
Gene ID: 16131238, Score: 18.884660487996918
Gene ID: 16129301, Score: 18.32374346124241
Gene ID: 16132076, Score: 18.117292583774987
Gene ID: 16128684, Score: 18.086704263941563
Gene ID: 16130583, Score: 17.884660487996918
Gene ID: 16131795, Score: 17.884660487996918
Gene ID: 16128026, Score: 17.18011637152309
Gene ID: 16129126, Score: 17.174167105191902
Gene ID: 49176132, Score: 17.034236844047427
Gene ID: 16132077, Score: 16.81427116010552
Gene ID: 16128462, Score: 16.635140888637608
Gene ID: 16131312, Score: 16.392807391667244
Gene ID: b4451, Score: 16.392807391667244
Gene ID: 16128105, Score: 16.303814095747995
Gene ID: 16128258, Score: 16.117292583774987
Gene ID: 16131247, Score: 16.00181536635505
Gene ID: 16131392, Score: 15.934701170496513
Gene ID: 16131884, Score: 15.93470117049651